<div dir="rtl">

# 04 — الـ logit lens: الموديل بيشوف إيه عند كل طبقة؟  ·  Watching the model's running guess, layer by layer

> *في نوتبوك ٠٣ سمّينا قطع الموديل واتفقنا إن الـ residual stream هو العمود الفقري اللي
> كل طبقة بتضيف عليه. النوتبوك ده بياخد الفكرة دي ويلعب بيها: لو احنا قدرنا نقرا الـ residual
> stream عند أي طبقة كأنها الطبقة الأخيرة، نشوف إيه؟ بنشوف توقّع الموديل وهو بيتطوّر مع العمق —
> أول لمحة "interpretability" حقيقية في المنهج ده.*
>
> nb03 named the parts of a small transformer and established that the residual stream
> is the model's spine. This notebook plays with one consequence of that: if we read
> the residual stream at *any* layer as if it were the final one, what do we see? We
> see the model's running best-guess evolve with depth — the first real interpretability
> peek in this curriculum.

**Prereqs.** nb02 (tokenizers) and nb03 (parts of the box).  
**Companion (next).** [`05-dialect-probe-masri.ipynb`](05-dialect-probe-masri.ipynb) — the first proper measurement: does the model encode dialect (MSA vs Masri) anywhere in its residual stream?

**A note on prompt choice.** For the lens to *show* anything, the model has to actually know the answer. `pythia-160m` knows English much better than Arabic, so we lead with an English factual prompt to see the lens working, then attempt a Masri version to see the same trick on a model that's out of its depth. The contrast itself is the lesson — and the bridge to nb05.

</div>

<div dir="rtl">

## 1. الإعداد  ·  Setup

</div>

In [ ]:
from __future__ import annotations

import os

import torch
from IPython.display import HTML, display
from transformer_lens import HookedTransformer

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("pythia-160m", device=DEVICE)
model.eval()
print(f"device: {DEVICE}  ·  layers: {model.cfg.n_layers}  ·  d_model: {model.cfg.d_model}")

<div dir="rtl">

## 2. الفكرة في سطر  ·  The idea in one line

في نوتبوك ٠٣ شفنا إن الموديل في الآخر بيـ"ضرب" الـ residual stream في مصفوفة الـ unembedding (`W_U`) عشان يطلع logits. السؤال: إيه اللي بيمنعنا إننا نضرب الـ residual stream في `W_U` عند **أي طبقة**، مش بس الأخيرة؟ مفيش حاجة بتمنعنا. والنتيجة بتقولنا: "لو الموديل اضطر يعمل توقّع دلوقتي، عند الطبقة دي، كان هيقول إيه؟"

ده هو الـ **logit lens**. اسمها كبير، فكرتها صغيرة.

From nb03: the model produces logits by multiplying the final residual stream by the unembedding matrix `W_U`. Question: what stops us from multiplying by `W_U` at *any* layer, not just the last? Nothing does. And the result tells us: "if the model had to commit to a prediction *right now*, at this depth, what would it say?"

That's the **logit lens**. Big name, small idea.

</div>

<div dir="rtl">

## 3. اللينس على جملة إنجليزية بسيطة  ·  The lens on a clean English prompt

نبدأ بحاجة الموديل عارفها كويس: "The capital of France is". الإجابة المتوقعة: "Paris". هنشغّل الموديل، نخزّن الـ residual stream عند كل طبقة، ونـ"نقراها" بـ `W_U` لكل طبقة على حدة.

We start with something the model knows well: "The capital of France is". Expected next token: "Paris". We run the model, cache the residual stream at every layer, and "read" each one through `W_U` independently.

</div>

In [ ]:
def logit_lens(prompt: str, top_k: int = 3) -> list[dict]:
    """For each layer's residual stream, project to vocab and report top-k tokens at the last position."""
    tokens = model.to_tokens(prompt)
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens)

    rows = []
    # resid_pre at layer 0 is the embedding; resid_post at layer L is the spine after layer L.
    # We include both endpoints so the trajectory starts "before any layer" and ends "after all layers".
    layers_to_read = [("embed", cache["resid_pre", 0])]
    for layer in range(model.cfg.n_layers):
        layers_to_read.append((f"L{layer}", cache["resid_post", layer]))

    for label, resid in layers_to_read:
        # Apply the model's final layer norm + unembedding the same way the real forward pass does.
        normed = model.ln_final(resid)
        logits = normed @ model.W_U + model.b_U  # (batch, n_tokens, d_vocab)
        last_pos_logits = logits[0, -1]
        probs = torch.softmax(last_pos_logits, dim=-1)
        topk = torch.topk(probs, k=top_k)
        rows.append({
            "layer": label,
            "top_tokens": [(model.to_string([i]), float(p)) for p, i in zip(topk.values, topk.indices)],
        })
    return rows


def print_trajectory(rows: list[dict]) -> None:
    print(f"{'layer':>6}  {'top-1':<20} {'top-2':<20} {'top-3':<20}")
    print("-" * 70)
    for row in rows:
        cells = [f"{tok!r:<14} {p:>4.0%}" for tok, p in row["top_tokens"]]
        print(f"{row['layer']:>6}  " + "  ".join(cells))


EN_PROMPT = "The capital of France is"
print(f"prompt: {EN_PROMPT!r}\n")
print_trajectory(logit_lens(EN_PROMPT))

<div dir="rtl">

*المتوقع:* عند الطبقات الأولى، التوقّع بيبقى عشوائي تمامًا (كلمات شائعة، مفيش معنى). كل ما نروح أعمق، الإجابة "Paris" بتبدأ تظهر — الأول كاحتمال صغير، بعدين تكبر، وفي الآخر بتبقى التوقّع الأول بثقة عالية. المسار ده هو "الموديل وهو بيفكّر"، مرئيًا.

*Expected:* at early layers the top-k is junk (common tokens, no signal). As we go deeper, "Paris" appears — first faintly, then grows, until it dominates. That trajectory *is* the model thinking, made visible.

</div>

<div dir="rtl">

## 4. اللينس على جملة ماصري  ·  The same lens on a Masri prompt

دلوقتي نجرّب نفس الحركة على جملة ماصري بسيطة: "عاصمة مصر هي". الإجابة الـ"صحيحة" (لو الموديل عارف): "القاهرة".

Now we try the same move on a simple Masri prompt: "عاصمة مصر هي" ("The capital of Egypt is"). The "right" answer, if the model knows it: "القاهرة" (Cairo).

</div>

In [ ]:
AR_PROMPT = "عاصمة مصر هي"
print(f"prompt: {AR_PROMPT!r}\n")
print_trajectory(logit_lens(AR_PROMPT))

<div dir="rtl">

*المتوقع:* المسار هيبقى مكسّر. الموديل ما اتدرّبش على عربي بشكل جدّي، فلا "القاهرة" ولا أي إجابة منطقية هتظهر بثقة. اللي هنشوفه هو "عشوائية محترمة" — توكنز شائعة، fragments عربية أو لاتينية، بدون مسار واضح.

**ده مش فشل في اللينس.** اللينس شغّال صح. اللي بيقوله: الموديل ده فعلاً ما عندوش الإجابة، فمفيش حاجة يرشّحها للسطح.

وده بالظبط السؤال اللي نوتبوك ٠٥ هيجاوب عليه بشكل أنضف: ممكن نقيس "الموديل عارف إيه عن العربي" حتى لو ما قدرش يجاوب؟ الإجابة (هتطلع معانا): آه، عن طريق الـ probes.

*Expected:* the trajectory will be broken. `pythia-160m` wasn't seriously trained on Arabic, so neither "القاهرة" nor any sensible answer will rise. What you'll see is dignified noise — common tokens, Arabic or Latin fragments, no clear arc.

**This is not a failure of the lens.** The lens is doing its job. What it's telling us is: this model genuinely doesn't have the answer to surface.

And that's the question nb05 picks up cleanly: can we measure what a model *knows* about a language even when it can't *answer* in that language? The answer (spoiler): yes, via probes.

</div>

<div dir="rtl">

## 5. اللينس بيقول إيه — وما بيقولش إيه  ·  What the lens does and doesn't say

اللينس **بيقول:** "لو ضربنا الـ residual stream عند الطبقة دي في مصفوفة الـ unembedding، الترجمة بتاعتها لـ vocab بتبقى إيه؟"  
اللينس **مش بيقول:** "الموديل بيـ"يفكّر" في الكلمة دي عند الطبقة دي."

الفرق مهم. الـ residual stream عند الطبقة الـ ٥ مش حالة "تفكير" نهائية — هو محطة في رحلة. الموديل اتدرّب بحيث المحطة الأخيرة بس هي اللي تتقرا بـ `W_U`. لما بنقرا محطة أبكر بنفس الـ lens، احنا بنعمل **تخمين معقول** عن نية الموديل — لكن مش قراءة دقيقة لـ"إيه اللي حصل في عقله".

في كتابات interpretability ده اسمه "the lens is a useful but imperfect read." نـستخدمه عشان يدّينا felt sense، مش عشان نـبني عليه claims قاطعة. الـ probes في نوتبوك ٠٥ بيدّونا أداة أدق.

The lens **says:** "if we project the residual stream at this layer through `W_U`, what does its vocabulary translation look like?"  
The lens **does not say:** "the model is *thinking* of this word at this layer."

This distinction matters. The residual stream at layer 5 is not a finished thought — it's a stop on a route. The model was trained so that only the final stop is meant to be read by `W_U`. Reading earlier stops through the same lens is a **plausible guess** at the model's evolving intent, not a precise readout of "what's in its head."

In interpretability writing this is called "the lens is a useful but imperfect read." Use it for felt sense, not for hard claims. The probes in nb05 give us a sharper instrument.

</div>

<div dir="rtl">

## 6. خلاصة + الخطوة الجاية  ·  Recap and what's next

**اللي اتعلمناه:**

- الـ logit lens = نفس الـ unembedding اللي الموديل بيستخدمه في الآخر، مطبّق على الـ residual stream عند طبقات أبكر.
- على جملة إنجليزية الموديل عارفها، اللينس بيوريك التوقّع بيتبلور مع العمق — حاجة عشوائية في البداية، تقرّب من الإجابة الصح في النهاية.
- على جملة ماصري الموديل ما عندوش معرفة كافية بيها، اللينس بيوري إن مفيش حاجة تتبلور — وده في حد ذاته معلومة مفيدة.
- اللينس أداة "إحساس" مش أداة قياس. نـستخدمه عشان نـفهم، مش عشان نـثبت.

**الخطوة الجاية:** نوتبوك ٠٥ — هل الموديل عارف إنه بيقرا ماصري؟ هندرّب probe خطّي بسيط على الـ residual stream عند كل طبقة عشان يصنّف MSA ضد ماصري. لو ال probe بيشتغل عند طبقة معينة، يبقى الموديل بيمثّل اللهجة في عمق ده — حتى لو مش قادر يـولّد عربي كويس.

**Recap:**

- The logit lens = the same unembedding the model uses at the end, applied to the residual stream at earlier layers.
- On an English prompt the model knows, the lens shows the prediction crystallising with depth — random early, settled by the end.
- On a Masri prompt the model doesn't really know, the lens shows nothing crystallising — and that's information too.
- The lens is a feel-instrument, not a measurement. Use it to *understand*, not to *prove*.

**Next:** notebook 05 — does the model know it's reading Masri? We'll train a small linear probe on the residual stream at every layer to classify MSA vs Masri. If the probe works at some layer, the model represents dialect at that depth — even if it can't generate Arabic well.

</div>